# 4b_GBM_MODEL — GBM Training & Artefact Export

Trains (or loads) three **HistGradientBoostingRegressor** models at
Q05 / Q50 / Q95 using optimised parameters from `gbm_params.json`.

| § | Stage | Notes |
|---|---|---|
| 1 | Config & imports | |
| 2 | Load reference parquet | |
| 3 | Load params & obs_sel | From `gbm_params.json` |
| 4 | Train/test split + scaler | Same seed/fraction as 4a |
| 5 | Fit GBM ×3 | Q05 quantile · Q50 MSE · Q95 quantile |
| 6 | Predict + conformal calibration | |
| 7 | Centre-bias correction spline | Cal-set only |
| 8 | Metrics | |
| 9 | Diagnostic figures | |
| 10 | Bundle artefacts | `gbm_artefacts.pkl` |

**Outputs**
- `output/models/gbm_q{05,50,95}_model.pkl`
- `output/models/gbm_artefacts.pkl`
- `output/models/gbm_metrics.csv`
- `fig/models/gbm_*.png`

> **Design note:** The Q50 model uses `loss='squared_error'` (equivalent
> to median when distribution is symmetric, but numerically more stable
> with sample_weight). Quantile crossing is rare with HistGBM but can
> occur — the correction spline is applied to Q05/Q95 independently so
> monotonicity is approximately preserved via spline extrapolation.

## 1. Imports & constants

In [2]:
import sys, json, pickle, time, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from scipy.interpolate import PchipInterpolator
from scipy.stats import entropy as scipy_entropy
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
warnings.filterwarnings('ignore')

# ── paths ──────────────────────────────────────────────────────────────────────
local_data = Path('data')
param_dir  = Path('output/sweeps')
model_dir  = Path('output/models')
fig_dir    = Path('fig/models')
model_dir.mkdir(parents=True, exist_ok=True)
fig_dir.mkdir(parents=True, exist_ok=True)

# ── shared constants ───────────────────────────────────────────────────────────
TEST_FRAC       = 0.20
RANDOM_SEED     = 42
TARGET_COL      = 'q'
Q_CLIP_MIN      = 0.0
QUANTILES       = [0.05, 0.25, 0.50, 0.75, 0.95]
QUANTILES_DENSE = np.linspace(0.02, 0.98, 49).tolist()
HIST_BIN_W      = 0.010
CONFORMAL_ALPHA = 0.10
SPLINE_PCTLS    = 200   # number of percentile knots for correction spline
FIG_DPI         = 150

from config import obs_model, q_clip_max, random_state
print(f'obs_model features : {len(obs_model)}')
print(f'q_clip_max         : {q_clip_max} W/m²')
print(f'Test fraction      : {TEST_FRAC}')


obs_model features : 21
q_clip_max         : 0.35 W/m²
Test fraction      : 0.2


## 2. Load reference data

In [4]:
df = pd.read_parquet(local_data / 'IHFC_obs.parquet').copy()
print(f'Raw rows: {len(df)}')

df = df.dropna(subset=obs_model)
print(f'After dropna : {len(df)} rows')

if 'weight' in df.columns:
    w_all = df['weight'].values.astype(np.float32)
    w_all = w_all / w_all.mean()
    print(f'Weights  min={w_all.min():.4f}  max={w_all.max():.4f}  mean={w_all.mean():.4f}')
else:
    w_all = np.ones(len(df), dtype=np.float32)
    print('WARNING: no weight column — using uniform weights')


Raw rows: 30848
After dropna : 30847 rows
Weights  min=0.1123  max=3.5262  mean=1.0000


## 3. Load sweep parameters

In [ ]:
from sklearn.ensemble import HistGradientBoostingRegressor

with open(param_dir / 'gbm_params.json') as fp:
    PARAMS = json.load(fp)

obs_sel = PARAMS.get('features', obs_model)
print(f'Loaded gbm_params.json')
print(f'  obs_sel ({len(obs_sel)} features)')
for k in ('max_iter','max_depth','learning_rate','min_samples_leaf',
          'max_features','l2_regularization'):
    print(f'  {k:22s} : {PARAMS.get(k)}')


## 4. Train/test split & StandardScaler

In [ ]:
X_all = df[obs_sel].values.astype(np.float32)
y_all = df[TARGET_COL].values.astype(np.float32)

assert np.isfinite(X_all).all(), 'NaNs in X_all — check obs_sel'
assert np.isfinite(y_all).all(), 'NaNs in y_all'

X_tr_raw, X_te_raw, y_tr, y_te, w_tr, w_te = train_test_split(
    X_all, y_all, w_all, test_size=TEST_FRAC, random_state=RANDOM_SEED)

scaler = StandardScaler().fit(X_tr_raw)
X_tr   = scaler.transform(X_tr_raw).astype(np.float32)
X_te   = scaler.transform(X_te_raw).astype(np.float32)

# Held-out calibration slice (last 10 % of train — no additional leakage)
cal_frac = float(PARAMS.get('cal_frac', 0.10))
n_cal    = max(int(len(X_tr) * cal_frac), 100)
X_cal, y_cal, w_cal = X_tr[-n_cal:], y_tr[-n_cal:], w_tr[-n_cal:]

print(f'Train  : {len(y_tr):,}    Test : {len(y_te):,}    Cal slice : {n_cal:,}')
print(f'Features used : {len(obs_sel)}')


## 5. Fit GBM (Q05 / Q50 / Q95)

Resumes from pickles if they exist. Each quantile is a separate model.

In [ ]:
quantile_losses = PARAMS.get('quantile_losses', [0.05, 0.50, 0.95])
gbm_models = {}
t0 = time.time()

for q in quantile_losses:
    path = model_dir / f'gbm_q{int(q*100):02d}_model.pkl'
    if path.exists():
        print(f'GBM Q{int(q*100):02d}: loading …')
        with open(path, 'rb') as fp:
            gbm_models[q] = pickle.load(fp)
    else:
        print(f'GBM Q{int(q*100):02d}: training …')
        p = dict(
            max_iter          = int(PARAMS['max_iter']),
            max_depth         = PARAMS.get('max_depth'),
            learning_rate     = float(PARAMS['learning_rate']),
            min_samples_leaf  = int(PARAMS['min_samples_leaf']),
            max_features      = PARAMS.get('max_features', 1.0),
            l2_regularization = float(PARAMS.get('l2_regularization', 0.0)),
            random_state      = RANDOM_SEED,
        )
        if q == 0.5:
            p['loss'] = 'squared_error'
        else:
            p['loss']     = 'quantile'
            p['quantile'] = q
        gbm_models[q] = HistGradientBoostingRegressor(**p).fit(
            X_tr, y_tr, sample_weight=w_tr)
        with open(path, 'wb') as fp:
            pickle.dump(gbm_models[q], fp)
        print(f'  saved → {path}')

print(f'Wall time: {(time.time()-t0)/60:.1f} min')


## 6. Test-set predictions + conformal calibration

In [ ]:
gbm_q05 = gbm_models[0.05].predict(X_te).astype(np.float32)
gbm_q50 = gbm_models[0.50].predict(X_te).astype(np.float32)
gbm_q95 = gbm_models[0.95].predict(X_te).astype(np.float32)

# conformal calibration on held-out cal slice
cal_05_gbm = gbm_models[0.05].predict(X_cal)
cal_95_gbm = gbm_models[0.95].predict(X_cal)
scores_gbm = np.maximum(cal_05_gbm - y_cal, y_cal - cal_95_gbm)
qhat_gbm   = np.quantile(scores_gbm,
                          (1 - CONFORMAL_ALPHA) * (1 + 1 / len(scores_gbm)))
gbm_conf_lo = np.clip(gbm_q05 - qhat_gbm, Q_CLIP_MIN, None)
gbm_conf_hi = np.clip(gbm_q95 + qhat_gbm, None, q_clip_max)
print(f'Conformal qhat = {qhat_gbm*1e3:.2f} mW/m²')


## 7. Correction spline helpers

In [ ]:
def weighted_percentile(vals, weights, pcts):
    """Weighted quantile (ignores NaN/inf)."""
    mask = np.isfinite(vals) & np.isfinite(weights)
    vs, ws = vals[mask], weights[mask]
    idx  = np.argsort(vs)
    cumw = np.cumsum(ws[idx]) / ws.sum()
    return np.interp(pcts / 100.0, cumw, vs[idx])

def build_correction_spline(y_pred_cal, y_true_cal, w_cal, n_pctls=SPLINE_PCTLS):
    """
    PCHIP spline mapping model predictions → empirical reference distribution.
    Fitted on calibration slice only (no test leakage).
    Addresses regression-to-the-mean / centre-of-distribution bias.
    """
    pctls      = np.linspace(1, 99, n_pctls)
    pred_p     = weighted_percentile(y_pred_cal, w_cal, pctls)
    true_p     = weighted_percentile(y_true_cal, w_cal, pctls)
    _, keep    = np.unique(pred_p, return_index=True)
    spline     = PchipInterpolator(pred_p[keep], true_p[keep], extrapolate=True)
    return spline, pred_p, true_p

def apply_spline(vals, spline):
    out    = np.full_like(vals, np.nan, dtype=np.float32)
    ok     = np.isfinite(vals)
    out[ok] = np.clip(spline(vals[ok]).astype(np.float32), Q_CLIP_MIN, q_clip_max)
    return out

print('Spline helpers defined.')


## 7a. Fit & apply correction spline

In [ ]:
print('Fitting GBM correction spline on calibration set …')
gbm_cal_q50 = gbm_models[0.50].predict(X_cal).astype(np.float32)
gbm_spline, gbm_pred_p, gbm_true_p = build_correction_spline(
    gbm_cal_q50, y_cal, w_cal)

gbm_q50_corr   = apply_spline(gbm_q50, gbm_spline)
gbm_q05_corr   = apply_spline(gbm_q05, gbm_spline)
gbm_q95_corr   = apply_spline(gbm_q95, gbm_spline)

fig, ax = plt.subplots(figsize=(5, 4))
ax.plot(gbm_pred_p*1e3, gbm_true_p*1e3, lw=1.5, color='#4CAF50', label='Spline')
ax.plot([0, q_clip_max*1e3], [0, q_clip_max*1e3], 'k--', lw=0.8, label='1:1')
ax.set_xlabel('Predicted Q [mW/m²]'); ax.set_ylabel('Corrected Q [mW/m²]')
ax.set_title('GBM centre-bias correction spline (cal set)')
ax.legend(); fig.tight_layout()
fig.savefig(fig_dir / 'gbm_correction_spline.png', dpi=FIG_DPI, bbox_inches='tight')
plt.show(); print(f'Saved {fig_dir}/gbm_correction_spline.png')


## 8. Metrics

In [ ]:
def empirical_entropy(vals, bin_width=HIST_BIN_W):
    bins   = np.arange(Q_CLIP_MIN, q_clip_max + bin_width, bin_width)
    counts, _ = np.histogram(vals[np.isfinite(vals)], bins=bins)
    probs  = counts / counts.sum()
    return float(scipy_entropy(probs[probs > 0]))

def mean_bias(y_true, y_pred):
    valid = np.isfinite(y_pred) & np.isfinite(y_true)
    return float(np.mean(y_pred[valid] - y_true[valid]))

def eval_metrics(y_true, y_pred, y_lo=None, y_hi=None, label=''):
    valid = np.isfinite(y_pred) & np.isfinite(y_true)
    r2    = r2_score(y_true[valid], y_pred[valid])
    rmse  = float(np.sqrt(mean_squared_error(y_true[valid], y_pred[valid])))
    mae   = float(mean_absolute_error(y_true[valid], y_pred[valid]))
    bias  = mean_bias(y_true, y_pred)
    picp  = np.nan
    piw   = np.nan
    if y_lo is not None and y_hi is not None:
        cov  = np.isfinite(y_lo) & np.isfinite(y_hi)
        picp = float(np.mean((y_true[cov] >= y_lo[cov]) & (y_true[cov] <= y_hi[cov])))
        piw  = float(np.mean(y_hi[cov] - y_lo[cov]))
    H = empirical_entropy(y_pred)
    print(f'{label:30s}  R²={r2:.4f}  RMSE={rmse*1e3:.2f} mW/m²  MAE={mae*1e3:.2f}  '
          f'Bias={bias*1e3:.2f}  PICP={picp:.3f}  PI_w={piw*1e3:.1f}  H={H:.3f}')
    return dict(label=label, r2=r2, rmse_mW=rmse*1e3, mae_mW=mae*1e3,
                bias_mW=bias*1e3, picp=picp, pi_width_mW=piw*1e3,
                shannon_H=H, nan_frac=float((~valid).mean()))

print('Metrics helpers defined.')


In [ ]:
H_obs   = empirical_entropy(y_te)
m_raw   = eval_metrics(y_te, gbm_q50,      y_lo=gbm_q05, y_hi=gbm_q95,
                        label='GBM Q50 raw')
m_corr  = eval_metrics(y_te, gbm_q50_corr, y_lo=gbm_q05_corr, y_hi=gbm_q95_corr,
                        label='GBM Q50 corrected')
m_conf  = eval_metrics(y_te, gbm_q50_corr, y_lo=gbm_conf_lo, y_hi=gbm_conf_hi,
                        label='GBM Q50 corr+conformal')
print(f'Observed entropy : {H_obs:.3f} nat')

metrics_df = pd.DataFrame([m_raw, m_corr, m_conf]).round(4)
metrics_df.to_csv(model_dir / 'gbm_metrics.csv', index=False)
print(f'Saved {model_dir}/gbm_metrics.csv')


## 9. Diagnostic figures

In [ ]:
def scatter_residuals_fig(rows, suptitle, savepath):
    """
    rows: list of (y_true, y_raw, y_corr, label, color)
    3-column figure: uncorrected | corrected | residual overlay
    """
    lim  = (Q_CLIP_MIN * 1e3, q_clip_max * 1e3)
    bins = np.linspace(lim[0], lim[1], 80)
    fig, axes = plt.subplots(len(rows), 3, figsize=(18, 5 * len(rows)))
    if len(rows) == 1: axes = axes[None, :]

    for i, (yt, yraw, ycorr, label, color) in enumerate(rows):
        yt_mW, yr_mW, yc_mW = yt*1e3, yraw*1e3, ycorr*1e3
        for j, (yp, cmap, alpha, lbl) in enumerate([
                (yr_mW, 'Oranges', 0.8, 'uncorr'),
                (yc_mW, 'Blues',   0.8, 'corr')]):
            ax = axes[i, j]
            h, xe, ye = np.histogram2d(yt_mW, yp, bins=bins)
            h = np.ma.masked_where(h == 0, h)
            ax.pcolormesh(xe, ye, h.T, cmap=cmap,
                          norm=plt.matplotlib.colors.LogNorm(),
                          rasterized=True)
            ax.plot(lim, lim, 'k--', lw=0.9, label='1:1')
            r2 = r2_score(yt_mW, yp)
            ax.set_title(f'{label} — {lbl}  R²={r2:.3f}', fontsize=9)
            ax.set_xlim(lim); ax.set_ylim(lim)
            ax.set_xlabel('Observed [mW/m²]'); ax.set_ylabel('Predicted [mW/m²]')
            ax.legend(fontsize=7)

        ax = axes[i, 2]
        rr = yr_mW - yt_mW
        rc = yc_mW - yt_mW
        ax.hist(rr, bins=60, color='#FF9800', alpha=0.5, edgecolor='none',
                label=f'uncorr  bias={np.mean(rr):.1f} σ={np.std(rr):.1f}')
        ax.hist(rc, bins=60, color=color, alpha=0.65, edgecolor='none',
                label=f'corr    bias={np.mean(rc):.1f} σ={np.std(rc):.1f}')
        ax.axvline(0, color='k', lw=0.9)
        ax.set_xlabel('Residual [mW/m²]'); ax.set_ylabel('Count')
        ax.set_xlim(-200, 200); ax.set_title(f'{label} — residuals', fontsize=9)
        ax.legend(fontsize=7)

    fig.suptitle(suptitle, fontsize=12, y=1.01)
    fig.tight_layout()
    fig.savefig(savepath, dpi=FIG_DPI, bbox_inches='tight')
    plt.show()
    print(f'Saved {savepath}')

print('Figure helper defined.')


In [ ]:
scatter_residuals_fig(
    rows=[(y_te, gbm_q50, gbm_q50_corr, 'GBM Q50', '#4CAF50')],
    suptitle='GBM — held-out test set (20%)',
    savepath=fig_dir / 'gbm_scatter_residuals.png',
)

fig, ax = plt.subplots(figsize=(7, 4))
pi_width = (gbm_q95_corr - gbm_q05_corr) * 1e3
ax.scatter(y_te * 1e3, pi_width, s=1, alpha=0.2, c='#4CAF50', rasterized=True)
ax.set_xlabel('Observed Q [mW/m²]'); ax.set_ylabel('PI90 width [mW/m²]')
ax.set_title(f'GBM corrected PI90 width  mean={pi_width.mean():.1f} mW/m²')
fig.tight_layout()
fig.savefig(fig_dir / 'gbm_pi_width.png', dpi=FIG_DPI, bbox_inches='tight')
plt.show(); print(f'Saved {fig_dir}/gbm_pi_width.png')


## 10. Bundle artefacts for `5_TARGETS`

In [ ]:
artefacts = dict(
    model        = 'GBM',
    obs_sel      = obs_sel,
    scaler       = scaler,
    gbm_models   = gbm_models,
    gbm_spline   = gbm_spline,
    qhat_gbm     = float(qhat_gbm),
    quantile_losses = list(quantile_losses),
    Q_CLIP_MIN   = Q_CLIP_MIN,
    q_clip_max   = float(q_clip_max),
    CONFORMAL_ALPHA = CONFORMAL_ALPHA,
    PARAMS       = PARAMS,
)
bundle_path = model_dir / 'gbm_artefacts.pkl'
with open(bundle_path, 'wb') as fp:
    pickle.dump(artefacts, fp)
print(f'Bundle saved → {bundle_path}')
print('Keys:', list(artefacts.keys()))
